# 07 · Free Colab TPU (v5e-1): can it train SinDDM faster than the T4? (Phase 2, optional)

**Runtime → Change runtime type → *v5e-1 TPU*** (the free-tier TPU; v2-8 was retired in 2025). Then *Run all*.
About 30–45 min, mostly XLA graph compilation on the first run.

**Why only SinDDM.** SinDDM training is the one compute bottleneck of this project (~7–8 h/image on an
A6000). Everything else stays on the T4:
- The paper's method has nothing to train, and its code is CUDA-only.
- LoRA takes only ~20–40 min/image on a T4.

**How.** PyTorch/XLA runs our SinDDM port with `--device xla`. That mode:
- never syncs device→host inside the training loop;
- cuts one XLA graph per step;
- writes checkpoints as CPU tensors, so a TPU-trained model loads and samples on the T4 or a CPU.

**Precision note.** By default, TPU matmuls use bf16 passes (`--xla_matmul_precision default`). The T4
trains in fp32. Both are normal for diffusion training, but report it if TPU-trained models are used.

**Decision rule.** Use the TPU for Phase-8 SinDDM training only if it is **≥ 2× faster** than the T4 at the
same image size *and* its loss trace looks like the T4's. Otherwise stay on the T4.

In [1]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK = "07_tpu_sinddm"
TORCH_XLA_VERSION = "2.8.0"   # torch and torch_xla must match; try "2.7.0" if this fails

In [2]:
import os, glob, subprocess
print("TPU_ACCELERATOR_TYPE =", os.environ.get("TPU_ACCELERATOR_TYPE"), "| /dev/accel*:", glob.glob("/dev/accel*") or glob.glob("/dev/vfio/*"))
assert os.environ.get("TPU_ACCELERATOR_TYPE") or glob.glob("/dev/accel*") or glob.glob("/dev/vfio/*"), \
    "No TPU: Runtime > Change runtime type > v5e-1 TPU (then rerun)."

TPU_ACCELERATOR_TYPE = v5e-1 | /dev/accel*: ['/dev/vfio/vfio', '/dev/vfio/0']


In [4]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

Mounted at /content/drive
project code at /content/project -> ['.git', '.gitignore', '2606.04299v1.pdf', 'PROJECT_PLAN.md', 'README.md', 'configs', 'docs', 'env', 'notebooks', 'patches', 'pyproject.toml', 'scripts', 'src', 'tests']


## Install PyTorch/XLA (matching torch) and smoke-test the TPU

In [5]:
TORCHVISION = {"2.8.0": "0.23.0", "2.7.0": "0.22.0", "2.6.0": "0.21.0"}[TORCH_XLA_VERSION]  # SinDDM imports torchvision
# The Colab TPU image now ships `torch_tpu` (a native backend for torch >= 2.11). torch auto-loads it on
# `import torch`, and it crashes with the older torch that torch_xla needs. We use torch_xla, so remove it
# and switch backend auto-loading off (this kernel and every subprocess / ! command inherit os.environ).
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"
!pip uninstall -q -y torch_tpu
!pip install -q "torch=={TORCH_XLA_VERSION}" "torchvision=={TORCHVISION}" "torch_xla[tpu]=={TORCH_XLA_VERSION}" -f https://storage.googleapis.com/libtpu-releases/index.html -f https://storage.googleapis.com/libtpu-wheels/index.html
probe = subprocess.run(["python", "-c",
    "import torch, torch_xla, torch_xla.core.xla_model as xm; d = xm.xla_device(); "
    "a = torch.randn(1024, 1024, device=d); b = (a @ a).sum(); xm.mark_step(); "
    "print('torch', torch.__version__, '| torch_xla', torch_xla.__version__, '| device', d, '| ok', float(b) == float(b))"],
    capture_output=True, text=True)
print(probe.stdout[-1500:], probe.stderr[-3000:])
assert probe.returncode == 0, "PyTorch/XLA does not work on this runtime -> try another TORCH_XLA_VERSION, or stay on the T4"

torch 2.8.0+cu128 | torch_xla 2.8.0 | device xla:0 | ok True
 <string>:1: DeprecationWarning: Use torch_xla.device instead
<string>:1: DeprecationWarning: Use torch_xla.sync instead



In [6]:
SINDDM = "/content/SinDDM"
if not os.path.exists(SINDDM):
    subprocess.run(["git", "clone", "-q", "https://github.com/fallenshock/SinDDM", SINDDM], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", SINDDM_COMMIT], check=True)
shutil.copytree(f"{PROJECT}/patches/sinddm_port", SINDDM, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("CHANGES.diff"))
!pip install -q einops ftfy scikit-image
print(subprocess.run(["git", "-C", SINDDM, "status", "--short"], capture_output=True, text=True).stdout)

 M SinDDM/functions.py
 M SinDDM/trainer.py
 M main.py



## Steady-state training speed on the TPU
Same method as notebook 05 (100 vs 400 steps, the difference ÷ 300), so compilation cost cancels out.

In [7]:
import time, json, re
from PIL import Image
DEVICE_ARGS = "--device xla --xla_matmul_precision default"
os.chdir(SINDDM)
def make_input(name, short_side=None):
    d = f"/content/sinddm_bench/{name}/"; os.makedirs(d, exist_ok=True)
    im = Image.open(f"{SINDDM}/datasets/balloons/balloons.png").convert("RGB")
    if short_side:
        w, h = im.size; s = short_side / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
    im.save(d + f"{name}.png"); return d, im.size

def train_seconds(scope, folder, image, steps):
    cmd = (f"python main.py --scope {scope} --mode train --dataset_folder {folder} --image_name {image} "
           f"--results_folder /content/sinddm_bench/results_{steps}/ --train_num_steps {steps} "
           f"--save_and_sample_every 1000000 --checkpoint_every_min 1000 --skip_final_sample --seed 0 {DEVICE_ARGS}")
    t0 = time.time()
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dt = time.time() - t0
    losses = re.findall(r"step:(\d+) loss:([0-9.eE+-]+)", out.stdout)
    if out.returncode != 0 or "training completed" not in out.stdout:
        print(out.stdout[-2000:], out.stderr[-3000:])
        raise RuntimeError(f"training failed ({scope}, {steps} steps)")
    return dt, losses

results = {"device": "TPU v5e-1 (PyTorch/XLA)", "device_args": DEVICE_ARGS}
for name, short in (("bench_native", None), ("bench_128", 128)):
    folder, size = make_input(name, short)
    t100, _ = train_seconds(name, folder, f"{name}.png", 100)
    t400, losses = train_seconds(name, folder, f"{name}.png", 400)
    per_1000 = (t400 - t100) / 300 * 1000
    results[name] = {"input_wh": list(size), "wall_100_steps_s": round(t100, 1), "wall_400_steps_s": round(t400, 1),
                     "steady_seconds_per_1000_steps": round(per_1000, 1),
                     "full_120k_schedule_hours": round(per_1000 * 120 / 3600, 1),
                     "loss_trace_400": [(int(a), round(float(b), 4)) for a, b in losses]}
    print(f"{name} {size}: {per_1000:.1f} s / 1000 steps  ->  full 120k schedule ~{per_1000*120/3600:.1f} h")
json.dump(results, open(f"{DRIVE_ROOT}/tables/phase2_sinddm_timing_tpu.json", "w"), indent=2)
print("saved", f"{DRIVE_ROOT}/tables/phase2_sinddm_timing_tpu.json")

bench_native (248, 186): 1764.8 s / 1000 steps  ->  full 120k schedule ~58.8 h
bench_128 (171, 128): 1296.9 s / 1000 steps  ->  full 120k schedule ~43.2 h
saved /content/drive/MyDrive/CS787/tables/phase2_sinddm_timing_tpu.json


## Compare with the T4 (from notebook 05) and decide

In [8]:
t4_path = f"{DRIVE_ROOT}/tables/phase2_sinddm_timing.json"
tpu = json.load(open(f"{DRIVE_ROOT}/tables/phase2_sinddm_timing_tpu.json"))
if not os.path.exists(t4_path):
    print("Run notebook 05 on a T4 first to get the GPU numbers; TPU numbers are saved.")
else:
    t4 = json.load(open(t4_path))
    for name in ("bench_native", "bench_128"):
        g, t = t4[name]["steady_seconds_per_1000_steps"], tpu[name]["steady_seconds_per_1000_steps"]
        print(f"{name}: T4 {g:.0f} s/1k steps | TPU {t:.0f} s/1k steps | speed-up x{g / t:.2f}")
        print("   T4  loss:", t4[name]["loss_trace_400"])
        print("   TPU loss:", tpu[name]["loss_trace_400"])
    speedups = [t4[n]["steady_seconds_per_1000_steps"] / tpu[n]["steady_seconds_per_1000_steps"]
                for n in ("bench_native", "bench_128")]
    print("\nDECISION:", "use the TPU for SinDDM training (Phase 8)" if min(speedups) >= 2
          else "stay on the T4 (TPU gain < 2x)")

Run notebook 05 on a T4 first to get the GPU numbers; TPU numbers are saved.


## Optional: precision check
If the TPU is faster, also time `--xla_matmul_precision highest` (≈ fp32, slower).
Then you can report how much of the gain comes from bf16 matmuls.

In [9]:
import time, json, re
from PIL import Image
DEVICE_ARGS = "--device xla --xla_matmul_precision highest"
os.chdir(SINDDM)
def make_input(name, short_side=None):
    d = f"/content/sinddm_bench/{name}/"; os.makedirs(d, exist_ok=True)
    im = Image.open(f"{SINDDM}/datasets/balloons/balloons.png").convert("RGB")
    if short_side:
        w, h = im.size; s = short_side / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
    im.save(d + f"{name}.png"); return d, im.size

def train_seconds(scope, folder, image, steps):
    cmd = (f"python main.py --scope {scope} --mode train --dataset_folder {folder} --image_name {image} "
           f"--results_folder /content/sinddm_bench/results_{steps}/ --train_num_steps {steps} "
           f"--save_and_sample_every 1000000 --checkpoint_every_min 1000 --skip_final_sample --seed 0 {DEVICE_ARGS}")
    t0 = time.time()
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dt = time.time() - t0
    losses = re.findall(r"step:(\d+) loss:([0-9.eE+-]+)", out.stdout)
    if out.returncode != 0 or "training completed" not in out.stdout:
        print(out.stdout[-2000:], out.stderr[-3000:])
        raise RuntimeError(f"training failed ({scope}, {steps} steps)")
    return dt, losses

results = {"device": "TPU v5e-1 (PyTorch/XLA, highest precision)", "device_args": DEVICE_ARGS}
for name, short in (("bench_native", None), ("bench_128", 128)):
    folder, size = make_input(name, short)
    t100, _ = train_seconds(name, folder, f"{name}.png", 100)
    t400, losses = train_seconds(name, folder, f"{name}.png", 400)
    per_1000 = (t400 - t100) / 300 * 1000
    results[name] = {"input_wh": list(size), "wall_100_steps_s": round(t100, 1), "wall_400_steps_s": round(t400, 1),
                     "steady_seconds_per_1000_steps": round(per_1000, 1),
                     "full_120k_schedule_hours": round(per_1000 * 120 / 3600, 1),
                     "loss_trace_400": [(int(a), round(float(b), 4)) for a, b in losses]}
    print(f"{name} {size}: {per_1000:.1f} s / 1000 steps  ->  full 120k schedule ~{per_1000*120/3600:.1f} h")
json.dump(results, open(f"{DRIVE_ROOT}/tables/phase2_sinddm_timing_tpu_highest.json", "w"), indent=2)
print("saved", f"{DRIVE_ROOT}/tables/phase2_sinddm_timing_tpu_highest.json")

KeyboardInterrupt: 

## Cross-device check: a TPU-trained checkpoint loads on CPU

In [10]:
import collections, torch
ck = sorted(glob.glob("/content/sinddm_bench/results_400/bench_128/model-last.pt"))[0]
with torch.serialization.safe_globals([collections.Counter]):
    data = torch.load(ck, map_location="cpu", weights_only=True)
devs = {v.device.type for v in data["model"].values()}
print("checkpoint step", data["step"], "| tensor devices:", devs)
assert devs == {"cpu"}

checkpoint step 400 | tensor devices: {'cpu'}


In [11]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")

saved /content/drive/MyDrive/CS787/env_logs/2026-11-13_1759_07_tpu_sinddm_freeze.txt
